## 1. Configuration ⚠️ Must Edit

> Change `OBS_BUCKET` below to your own bucket name; when switching regions, update `OBS_ENDPOINT` to match.
> If the Notebook is bound to an OBS agency, leave AK/SK empty (moxing then uses the agency automatically);
> otherwise fill in your own AK/SK temporarily — **clear them as soon as you are done, and never commit real keys to the repository or share them**.


In [ ]:
# ==================== ⚠️ MUST EDIT ====================
OBS_BUCKET   = "<your-bucket-name>"                    # Your OBS bucket name
OBS_PREFIX   = "models"                                # OBS storage path prefix (= the first half of the server-side OBS_KEY)
OBS_ENDPOINT = "obs.cn-north-4.myhuaweicloud.com"      # OBS endpoint (update when switching regions; the deployment-side OBS_ENDPOINT must match)

# IAM access keys (Huawei Cloud console → My Credentials → Access Keys)
# ⚠️ Leave empty if the Notebook is bound to an OBS agency (moxing uses the agency automatically); fill in your own only when there is no agency
# ⚠️ Clear real AK/SK as soon as you are done — never commit them to the repository or share them
ACCESS_KEY_ID     = ""    # ← fill in your AK (or leave empty to use the agency)
SECRET_ACCESS_KEY = ""    # ← fill in your SK (or leave empty to use the agency)
# ==========================================================

import os
from pathlib import Path

# Working directory: a ModelArts Notebook uses its built-in writable directory; a local run falls back to the current directory
WORK_DIR = "/home/ma-user/work/xgb_train" if Path("/home/ma-user").exists() else "."

assert "<" not in OBS_BUCKET, "Please replace <your-bucket-name> with the actual bucket name first"

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
OLD_DIR = WORK / "model_out" / "old"
NEW_DIR = WORK / "model_out" / "new"
OLD_DIR.mkdir(parents=True, exist_ok=True)
NEW_DIR.mkdir(parents=True, exist_ok=True)

# Keep both model sets locally (for comparison verification)
OLD_MODEL_LOCAL = OLD_DIR / "xgboost_breast_cancer.json"
NEW_MODEL_LOCAL = NEW_DIR / "xgboost_breast_cancer.json"

# OBS has a single target path (no old/new subdirectories; switching happens via ACTIVE_MODEL in §7)
ACTIVE_MODEL_OBS = f"obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json"

print(f"Working directory: {WORK}")
print(f"OBS bucket:        {OBS_BUCKET}")
print(f"OBS target path:   {ACTIVE_MODEL_OBS}")


## 2. Install Dependencies + Imports

A ModelArts Notebook ships with `xgboost`, `scikit-learn`, and `pandas`.
We additionally verify that `moxing` (Huawei Cloud's OBS operations library) is available.


In [ ]:
# === Fix the pandas ABI conflict on ModelArts Notebooks ===
# Root cause: ~/modelarts-dev/modelarts-sdk/ bundles a pandas copy that gets moved to the
# front of sys.path; its C extensions do not match the environment's numpy version, causing
#   ValueError: numpy.dtype size changed (Expected 96, got 88)
# Fix: remove that directory from sys.path and purge the already-cached pandas modules,
# forcing a fallback to the healthy pandas inside conda site-packages that matches numpy.
import sys as _sys

_BAD_FRAGMENTS = ("modelarts-dev/modelarts-sdk", "modelarts-dev\\modelarts-sdk")
_original_path = list(_sys.path)
_sys.path = [p for p in _sys.path if not any(frag in p.replace("\\", "/") for frag in _BAD_FRAGMENTS)]
if len(_sys.path) != len(_original_path):
    removed = set(_original_path) - set(_sys.path)
    print(f"[path-fix] removed from sys.path: {removed}")

# Purge cached modules that may have been polluted by the broken pandas
for _mod_name in list(_sys.modules):
    if _mod_name == "pandas" or _mod_name.startswith("pandas."):
        _mod = _sys.modules[_mod_name]
        _mod_file = getattr(_mod, "__file__", "") or ""
        if any(frag in _mod_file.replace("\\", "/") for frag in _BAD_FRAGMENTS):
            del _sys.modules[_mod_name]
            print(f"[path-fix] unloaded cached module: {_mod_name}")
# === end of path-fix ===

# === Install missing dependencies (the ModelArts image may not preinstall scikit-learn / xgboost) ===
import subprocess as _sp
def _pip_install(*pkgs):
    print(f"[install] pip install {' '.join(pkgs)} ...")
    _sp.check_call([_sys.executable, "-m", "pip", "install", "--quiet", *pkgs])

for _pkg, _import_name in [("scikit-learn", "sklearn"), ("xgboost", "xgboost")]:
    try:
        __import__(_import_name)
        print(f"[install] {_pkg} already installed, skipping")
    except ImportError:
        _pip_install(_pkg)
# === end of install ===

import json
import shutil
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

print(f"pandas location: {pd.__file__}")
print(f"pandas version:  {pd.__version__}")
print(f"numpy  version:  {np.__version__}")

# ModelArts' built-in OBS operations library
try:
    import moxing as mox
    print(f"moxing version: {mox.__version__ if hasattr(mox, '__version__') else 'unknown'}")
    # If AK/SK are provided, set up authentication (otherwise the Notebook agency is used)
    if ACCESS_KEY_ID and SECRET_ACCESS_KEY:
        import moxing.framework.content_db as content_db
        content_db.configure_obs_credentials(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            endpoint=OBS_ENDPOINT,
        )
        print("moxing authentication configured with AK/SK")
    else:
        print("No AK/SK provided; the Notebook agency will be used for authentication")
    HAS_MOXING = True
except ImportError:
    HAS_MOXING = False
    print("⚠️ moxing unavailable; will try esdk-obs-python as a fallback")

print(f"\nxgboost: {__import__('xgboost').__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")


## 3. Connect to MRS Hive (Kerberos-secured cluster)

Training data now comes from the MRS Hive table `breast_cancer` (569 rows x 31 cols, schema in `hive_export/breast_cancer_hive.sql`) instead of the bundled sklearn dataset.

The following 6 cells are copied from cells 1–6 of `hive_export/modelarts_hive_conn_EN.ipynb` (the battle-tested recipe; troubleshooting and rationale live in that notebook and in `hive_export/MRS_RUN.md`, `docs/adr/0002`):

> ⚠️ After a kernel restart, cells 1 and 3–4 must be re-run (PATH / KRB5_CONFIG live only in memory); after the 24h ticket expiry, re-run cell 5 (re-enter the password).

In [ ]:
# ================== 1. Connection settings (measured values; edit here for another cluster) ==================
HIVE_HOST = "10.0.0.15"    # HiveServer2 internal IP (master1)
HIVE_PORT = 21066          # HiveServer2 Thrift port
DATABASE  = "default"
USERNAME  = "hhx"          # MRS business user

REALM    = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"  # MRS system domain (Realm)
SPN_HOST = "hadoop." + REALM.lower()   # measured-correct SPN middle part (haddop_ variant is wrong)

KDC_HOSTS = ["10.0.0.15", "10.0.0.51"]  # list both masters for fault tolerance
KDC_PORT  = 21732                        # Huawei MRS dedicated KDC port, NOT 88!

print("principal =", f"hive/{SPN_HOST}@{REALM}")


In [ ]:
# ================== 2. Network probing (a missing security-group rule surfaces here fast) ==================
import socket

def probe(host, port, name, timeout=5):
    s = socket.socket(); s.settimeout(timeout)
    try:
        s.connect((host, port)); print(f"[OK]   {name} {host}:{port} reachable")
        return True
    except Exception as e:
        print(f"[FAIL] {name} {host}:{port} unreachable: {e}")
        return False
    finally:
        s.close()

net_ok = probe(HIVE_HOST, HIVE_PORT, "HiveServer2")
for k in KDC_HOSTS:
    net_ok &= probe(k, KDC_PORT, "KDC")

assert net_ok, (
    "Network unreachable: confirm the notebook is in the same VPC as MRS, and the\n"
    "security group allows 21066 and 21732 (TCP+UDP); 21066 alone is not enough - kinit also needs the KDC on 21732."
)


In [ ]:
# ================== 3. Environment setup (idempotent; auto-adapts to three environments; live progress) ==================
# Goal: kinit binary + cyrus sasl (with the GSSAPI plugin in place) + pure-python pyhive etc.
#   * The cluster enforces qop=auth-conf, so cyrus sasl is mandatory; pure-sasl+pykerberos
#     was measured to fail at the encrypted-wrap stage with "Invalid token was supplied"
#     (see ADR-0002).
# Adaptation order (the [env] line tells you which branch fired):
#   A. root             -> apt install gcc/g++/krb5-user/headers + GSSAPI plugin, pip-build sasl
#   B. ma-user + sudo   -> same as A, with sudo -n in front of apt
#   C. no root (common) -> conda-forge prebuilt: krb5 (ships kinit) + sasl, no compiler needed
import collections, importlib, os, re, shutil, subprocess, sys, threading, time

def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

# ---- live-progress runner: print key lines immediately (with elapsed time),
#      heartbeat during quiet periods, replay tail output on failure ----
_BAR = re.compile(r"^\W*\[\W*\d+%\W*\]\W*$")        # apt's "[ 12%]" progress bars (noise)
_HOT = re.compile(r"solving|collecting|downloading|extracting|preparing|executing|"
                  r"transaction|unpacking|setting up|processing|fetched|^get|^hit|"
                  r"building wheel|successfully|installed|nothing to do|all requested|"
                  r"error|fail|conflict|warn", re.I)     # progress/result lines worth showing

def run_stream(cmd, note=None, heartbeat=20):
    """Stream an external command. Return code 0 = success; on failure replay the last 40 lines."""
    if note: print(f"[run] {note}", flush=True)
    t0, tail = time.time(), collections.deque(maxlen=40)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    stop = threading.Event()
    def _beat():                                          # prove "still alive" during long silences
        quiet = time.time()
        while not stop.wait(2):
            if time.time() - quiet >= heartbeat:
                print(f"   ... {int(time.time()-t0)}s still running ({cmd[0]} silent, that is normal)", flush=True)
                quiet = time.time()
    th = threading.Thread(target=_beat, daemon=True); th.start()
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line)
        if line and not _BAR.match(line) and _HOT.search(line):
            print(f"[{int(time.time()-t0):>3}s] {line}", flush=True)
    rc = proc.wait(); stop.set(); th.join(timeout=1)
    if rc != 0:
        print("---- tail of command output (up to 40 lines) ----")
        print("\n".join(t for t in tail if t.strip()) or "(no output)")
    return rc

# conda's kinit lives in sys.prefix/bin: after a kernel restart PATH may not include
# it, so prepend it here — otherwise need_kinit is misdetected even though deps are
# already installed, triggering a pointless conda solve (~5 minutes measured here)
if os.path.isfile(os.path.join(sys.prefix, "bin", "kinit")):
    os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
need_kinit, need_sasl = shutil.which("kinit") is None, not have("sasl")

# --- 3.1 system layer ---
if need_kinit or need_sasl:
    apt, env_name = None, "no root (conda branch)"
    if os.geteuid() == 0:
        apt, env_name = ["apt-get"], "root"
    else:
        sudo_ok = subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0
        if sudo_ok:
            apt, env_name = ["sudo", "-n", "apt-get"], "ma-user + passwordless sudo"
    print(f"[env] {env_name}", flush=True)

    if apt is not None:
        # libsasl2-modules-gssapi-mit / libsasl2-modules = cyrus GSSAPI plugins (required!)
        if run_stream(apt + ["update"], "apt-get update") != 0:
            raise SystemExit("[FAIL] apt-get update failed")
        if run_stream(apt + ["install", "-y", "gcc", "g++", "krb5-user",
                             "libkrb5-dev", "libsasl2-dev",
                             "libsasl2-modules-gssapi-mit", "libsasl2-modules"],
                      "apt install toolchain + krb5 + cyrus sasl (first run ~1-2 min)") != 0:
            raise SystemExit("[FAIL] apt install failed")
    else:
        # no root: ModelArts ships anaconda; conda-forge has prebuilt krb5 and sasl
        conda = shutil.which("conda")
        assert conda, "[FAIL] conda not found — please report this error to the maintainers"
        print("[env] no root — using the conda-forge prebuilt path", flush=True)
        # --prefix sys.prefix: install explicitly into the current kernel env, not base
        # --override-channels: use only the channels given here — if the instance's
        #   condarc points at dead mirror channels (e.g. TUNA anaconda/pkgs/free,
        #   no longer synced, 404), they would fail too without this flag
        base = [conda, "install", "-y", "--override-channels", "--prefix", sys.prefix]
        attempts = [
            (base + ["-c", "conda-forge", "krb5", "sasl"],
             "conda install krb5 + sasl (conda-forge, bypassing dead mirror channels; solve+download 1-3 min)"),
            (base + ["-c", "https://conda.anaconda.org/conda-forge", "krb5", "sasl"],
             "conda retry (direct official conda-forge, may be slower)"),
        ]
        rc = 1
        for cmd, note in attempts:
            rc = run_stream(cmd, note)
            if rc == 0:
                break
        if rc != 0:
            raise SystemExit("[FAIL] conda install failed on both sources (instance mirrors down?); "
                             "fallback: offline wheels via OBS, or report the output above to maintainers")
        # conda's kinit lives in $CONDA_PREFIX/bin — put it on PATH for cell 5
        os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
        print("[tip] If the 3.3 self-check below fails to import (kernel unaware of freshly installed conda "
              "packages), restart the kernel and re-run cells 1, 3, 4 — installed pieces are skipped automatically")
else:
    print("[env] system dependencies already in place (kinit + sasl), skipping installation")

# --- 3.2 python packages (pure python, pip is enough) ---
PIP_PKGS = [p for p, m in (("pyhive", "pyhive"), ("thrift", "thrift"),
                           ("thrift-sasl", "thrift_sasl"), ("sasl", "sasl"))
            if not have(m)]
if PIP_PKGS:
    if run_stream([sys.executable, "-m", "pip", "install"] + PIP_PKGS,
                  f"pip install {PIP_PKGS}") != 0:
        raise SystemExit("[FAIL] pip install failed")

# --- 3.3 self-check: imports + cyrus GSSAPI plugin in place (hard prerequisite for auth-conf) ---
# Note: the cyrus sasl package has no "list mechanisms" API (available_mechs belongs
# to pure-sasl; using it raises AttributeError). Use a functional probe instead:
# actually init + start GSSAPI once — the same code path cell 6 uses at connect time
# (pyhive.get_sasl_client -> setAttr+init; thrift_sasl.open -> start):
#   start succeeds                                 -> plugin in place (and a ticket exists)
#   "No worthy mechs" / "No mechanism available"   -> plugin missing (deps incomplete, fatal)
#   GSSAPI credential error (no ticket, etc.)      -> plugin in place; works after cell 5 kinit
import glob
from pyhive import hive
from pyhive.hive import get_installed_sasl
import thrift_sasl
import sasl as cyrus_sasl

_p = cyrus_sasl.Client()
_p.setAttr("host", SPN_HOST)          # SPN middle part from cell 1, as the SASL-layer parameter
_p.setAttr("service", "hive")
assert _p.init(), f"cyrus sasl init failed: {_p.getError()!r}"
_ok, _mech, _resp = _p.start("GSSAPI")
_err = _p.getError()
_err = _err.decode("utf-8", "replace") if isinstance(_err, bytes) else (_err or "")
if _ok:
    print("[OK] python dependencies ready; GSSAPI plugin usable; kinit =", shutil.which("kinit"))
elif "worthy mechs" in _err.lower() or "no mechanism available" in _err.lower():
    for _pat in (os.path.join(sys.prefix, "lib*", "sasl2", "*"),
                 "/usr/lib/*/sasl2/*", "/usr/lib64/sasl2/*"):
        for _h in glob.glob(_pat):
            if "gssapi" in os.path.basename(_h).lower():
                print("  gssapi plugin file:", _h)
    raise SystemExit(
        f"cyrus sasl is missing the GSSAPI plugin ({_err})\n"
        "root/sudo env: check that libsasl2-modules-gssapi-mit got installed;\n"
        "conda env: send the output of !ls $CONDA_PREFIX/lib/sasl2/ to the maintainers")
else:
    print("[OK] python dependencies ready; GSSAPI plugin in place (no ticket yet — effective "
          f"after kinit in cell 5; probe info: {_err.splitlines()[0] if _err else '-'})")
    print("kinit =", shutil.which("kinit"))


In [ ]:
# ================== 4. Generate krb5.conf and make it effective ==================
# dns_canonicalize_hostname=false is the key: the SPN middle part hadoop.xxx is a
# "fake domain" that does not exist in DNS — the Kerberos client must be stopped
# from resolving it, so it is used verbatim as the SPN.
# udp_preference_limit=1 forces AS/TGS requests onto TCP — matching the TCP port probed in cell 2.
import os
from pathlib import Path

KRB5_FILE = Path.cwd() / "krb5.conf"
lines = [
    "[libdefaults]",
    f"    default_realm = {REALM}",
    "    dns_canonicalize_hostname = false",   # <- key
    "    rdns = false",
    "    udp_preference_limit = 1",
    "",
    "[realms]",
    f"    {REALM} = {{",
    *[f"        kdc = {h}:{KDC_PORT}" for h in KDC_HOSTS],
    f"        admin_server = {KDC_HOSTS[0]}:{KDC_PORT}",
    "    }",
    "",
    "[domain_realm]",
    f"    .{REALM.lower()} = {REALM}",
    f"    {SPN_HOST} = {REALM}",
    f"    .{SPN_HOST} = {REALM}",
    "",
]
KRB5_FILE.write_text("\n".join(lines), encoding="utf-8")
os.environ["KRB5_CONFIG"] = str(KRB5_FILE)   # both kinit and cyrus-sasl read this later
print(f"[OK] generated {KRB5_FILE} and set KRB5_CONFIG\n")
print("\n".join(lines))


In [ ]:
# ================== 5. kinit to obtain the user ticket (TGT, valid 24h) ==================
# Skip if a valid ticket exists; otherwise prompt for the password (getpass — no
# plaintext left in code). kinit may come from apt (krb5-user, /usr/bin) or conda
# (krb5, $CONDA_PREFIX/bin) — adapt automatically.
import getpass, shutil, subprocess

KINIT = shutil.which("kinit") or os.path.join(sys.prefix, "bin", "kinit")
KLIST = shutil.which("klist") or os.path.join(sys.prefix, "bin", "klist")

def _run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True,
                          env={**os.environ, "KRB5_CONFIG": str(KRB5_FILE)}, **kw)

r = _run([KLIST])
if r.returncode == 0 and "krbtgt" in r.stdout:
    print("[OK] a valid ticket already exists, skipping kinit:")
    print("\n".join(r.stdout.splitlines()[:4]))
else:
    principal = f"{USERNAME}@{REALM}"
    pw = getpass.getpass(f"Password for {principal}: ")
    r = _run([KINIT, principal], input=pw + "\n")
    assert r.returncode == 0, f"[FAIL] kinit failed (wrong password / KDC unreachable?): {r.stderr.strip()}"
    print(f"[OK] kinit succeeded: {principal}")


In [ ]:
# ================== 6. Connect to HiveServer2 (core: decouple TCP address from SPN) ==================
# With auth=KERBEROS pyhive uses the TCP host directly as the SPN host -> guaranteed
# mismatch (it would ask the KDC for hive/10.0.0.15@REALM, but the cluster registers
# a fixed-string SPN). The official escape hatch is thrift_transport=...: the TCP
# layer connects to the internal IP while the SASL-layer host carries the SPN middle part.
# get_installed_sasl automatically prefers the cyrus sasl package once installed
# (mandatory for qop=auth-conf).
from thrift.transport import TSocket

def make_transport():
    tcp = TSocket.TSocket(HIVE_HOST, HIVE_PORT)
    tcp.setTimeout(30000)
    sasl_factory = lambda: get_installed_sasl(
        host=SPN_HOST, sasl_auth="GSSAPI", service="hive")
    return thrift_sasl.TSaslClientTransport(sasl_factory, "GSSAPI", tcp)

PRINCIPAL = f"hive/{SPN_HOST}@{REALM}"
print("Trying SPN:", PRINCIPAL)
conn = hive.connect(thrift_transport=make_transport(),
                    database=DATABASE, username=USERNAME)
print("[OK] connected! Effective SPN =", PRINCIPAL)


## 4. Load Data from Hive + Define the Sample

Reads the dataset from the MRS Hive table `breast_cancer` (replacing sklearn's `load_breast_cancer`), restores sklearn's spaced feature names from Hive's underscored columns, then embeds the test sample from `sample_request.json` (used to verify old/new models give different predictions).

> Data is fetched in small batches (`cur.arraysize = 5`): fetching the whole table at once triggers a large-frame decode bug in some libsasl2 builds (see the comment inside the cell).


In [ ]:
# === Read the breast cancer dataset from Hive (replaces sklearn's load_breast_cancer) ===
# Table schema: hive_export/breast_cancer_hive.sql - 569 rows x 31 cols (30 features + target)
cur = conn.cursor()
# Pitfall (measured): pyhive defaults to arraysze=10000, so fetchall packs all 569
# rows into one huge SASL-encrypted frame; some libsasl2 builds (known regression
# in 2.1.28) cannot decode large frames and raise
#   TTransportException: sasl_decode ... Unable to find a callback: 32775
# Fix: fetch in small batches of 5 - the same reason LIMIT 5 in the conn notebook is stable.
cur.arraysize = 5
cur.execute("SELECT * FROM breast_cancer")
cols = [d[0].split(".")[-1] for d in cur.description]   # strip any db.table. prefix
try:
    rows = cur.fetchall()
except Exception as e:
    if "sasl_decode" in str(e) or "32775" in str(e):
        msg = """sasl decode still failing with small batches (libsasl2 2.1.28 known regression).
Fix: run the two lines below in a new cell, then Kernel - Restart Kernel and re-run from the top:
  import subprocess
  subprocess.run(['conda', 'install', '-y', '-c', 'conda-forge',
                  '--override-channels', 'libsasl2=2.1.27'], check=True)"""
        raise SystemExit(msg) from e
    raise
finally:
    cur.close()
print(f"fetched {len(rows)} rows")

df_hive = pd.DataFrame(rows, columns=cols)
# Hive columns use underscores (mean_radius); restore sklearn's spaced style
# ("mean radius") so feature names match sample_request.json / app.py
df_hive.columns = [c.replace("_", " ") for c in df_hive.columns]

X = df_hive.drop(columns=["target"])
y = df_hive["target"].astype(int)
print(f"Hive breast_cancer: {X.shape[0]} samples, {X.shape[1]} features")

# === Data validation ===
# Feature names are borrowed from sklearn only (not its data) to guarantee the
# same feature order as the inference service
from sklearn.datasets import load_breast_cancer
FEATURE_NAMES = list(load_breast_cancer().feature_names)
assert list(X.columns) == FEATURE_NAMES, f"columns mismatch sklearn: {list(X.columns)[:3]} ..."
assert X.shape == (569, 30), f"expected 569x30, got {X.shape}"
assert set(y.unique()) <= {0, 1}, f"unexpected target values: {sorted(y.unique())}"
print("[OK] validation passed: 569x30, names match sklearn, target in {0,1}")

# === Embed the test sample (from sample_request.json) ===
sample_row = {
    "mean radius": 17.99, "mean texture": 10.38, "mean perimeter": 122.8,
    "mean area": 1001.0, "mean smoothness": 0.1184, "mean compactness": 0.2776,
    "mean concavity": 0.3001, "mean concave points": 0.1471,
    "mean symmetry": 0.2419, "mean fractal dimension": 0.07871,
    "radius error": 1.095, "texture error": 0.9053, "perimeter error": 8.589,
    "area error": 153.4, "smoothness error": 0.006399,
    "compactness error": 0.04904, "concavity error": 0.05373,
    "concave points error": 0.01587, "symmetry error": 0.03003,
    "fractal dimension error": 0.006193, "worst radius": 25.38,
    "worst texture": 17.33, "worst perimeter": 184.6, "worst area": 2019.0,
    "worst smoothness": 0.1622, "worst compactness": 0.6656,
    "worst concavity": 0.7119, "worst concave points": 0.2654,
    "worst symmetry": 0.4601, "worst fractal dimension": 0.1189,
}
sample_df = pd.DataFrame([sample_row], columns=FEATURE_NAMES)
print(f"test sample: {len(sample_row)} features")


## 5. Training Function

Train → evaluate → save locally → print the sample's predicted value.
The old and new models give different predictions for the same sample — that difference
is exactly the criterion for the hot swap verification later on.


In [ ]:
def train_and_save(params, random_state, output_path, label):
    """Train a model, evaluate and save it, and return the sample's prediction probability."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y,
    )
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=random_state,
        **params,
    )
    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    pred = float(model.predict_proba(sample_df)[0, 1])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(output_path))
    size = output_path.stat().st_size

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Hyperparams: {params}")
    print(f"  Accuracy:    {acc:.4f}  AUC: {auc:.4f}")
    print(f"  Sample prediction: {pred:.16f}")
    print(f"  Saved locally:     {output_path} ({size:,} bytes)")
    return pred


## 6. Train the OLD Model (baseline)

100 trees, shallow depth, high learning rate.


In [ ]:
old_pred = train_and_save(
    params=dict(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
    ),
    random_state=42,
    output_path=OLD_MODEL_LOCAL,
    label="OLD MODEL (baseline)",
)


## 7. Train the NEW Model (different hyperparameters)

250 trees, deeper trees, low learning rate, with regularization.


In [ ]:
new_pred = train_and_save(
    params=dict(
        n_estimators=250, max_depth=6, learning_rate=0.01,
        subsample=0.6, colsample_bytree=0.5,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0, gamma=0.5,
    ),
    random_state=2024,
    output_path=NEW_MODEL_LOCAL,
    label="NEW MODEL (updated)",
)


## 8. Upload the Model to OBS 🚀

Upload the trained model to the **single target path** on OBS (the inference service
only ever reads from this path):

- `obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json`

> The inference service (`app.py`, environment variables `OBS_BUCKET` + `OBS_KEY`) reads
> the model from this path.
> To switch between old / new: change `ACTIVE_MODEL` below and re-run this cell.


In [ ]:
# Choose which model set to upload to OBS (set to "new" to switch to the new model)
ACTIVE_MODEL = "old"   # "old" or "new"

LOCAL_TO_UPLOAD = OLD_MODEL_LOCAL if ACTIVE_MODEL == "old" else NEW_MODEL_LOCAL
print(f"Current selection: {ACTIVE_MODEL} model")
print(f"Local file:        {LOCAL_TO_UPLOAD} ({LOCAL_TO_UPLOAD.stat().st_size:,} bytes)")
print(f"OBS target:        {ACTIVE_MODEL_OBS}")
print()

def upload_to_obs(local_path, obs_uri, label=""):
    """Upload a single file to OBS (overwrites)."""
    tag = f" [{label}]" if label else ""
    print(f"  Upload{tag}: {local_path} → {obs_uri}")

    if HAS_MOXING:
        mox.file.copy(str(local_path), obs_uri)
    else:
        # fallback: esdk-obs-python
        from obs import ObsClient
        assert ACCESS_KEY_ID and SECRET_ACCESS_KEY, "AK/SK are required when moxing is unavailable"
        client = ObsClient(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            server=f"https://{OBS_ENDPOINT}",
        )
        key = obs_uri.replace(f"obs://{OBS_BUCKET}/", "")
        resp = client.putFile(OBS_BUCKET, key, str(local_path))
        assert resp.status < 300, f"Upload failed: status={resp.status}"
        client.close()

    size = local_path.stat().st_size
    print(f"    ✅ Done ({size:,} bytes)")

upload_to_obs(LOCAL_TO_UPLOAD, ACTIVE_MODEL_OBS, ACTIVE_MODEL.upper())
print(f"\n🚀 Upload complete!")
print(f"   The inference service app.py reads the model from {ACTIVE_MODEL_OBS}.")
print(f"   To switch models: set ACTIVE_MODEL to 'new' and re-run this cell.")
